# Week 1 Lab: Finance Fundamentals with Python

**BUS 696: Generative AI in Finance**  
**Professor Jonathan Hersh**

---

## Learning Objectives

By the end of this lab, you will be able to:
1. Download and explore real stock market data using `yfinance`
2. Understand OHLCV data (Open, High, Low, Close, Volume)
3. Build candlestick charts to visualize price action
4. Calculate daily returns and interpret summary statistics
5. Compute and plot moving averages as trading signals
6. Calculate the Sharpe Ratio to evaluate risk-adjusted performance
7. Compare multiple stocks side by side
8. Write effective prompts for financial analysis tasks

---

## Part 1: Setting Up and Getting Data

In [ ]:
# Install required packages (run once, then you can comment this out)
# !pip install yfinance pandas matplotlib plotly

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import numpy as np

# Make matplotlib plots look nicer
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries loaded!")

### Downloading Stock Data

We use the **yfinance** library to download free stock data from Yahoo Finance.

Let's start by downloading one year of Apple (AAPL) data.

In [ ]:
# Download Apple stock data for the past year
apple = yf.download('AAPL', period='1y')

# yfinance returns a MultiIndex when downloading even a single ticker.
# We flatten the columns so they're just 'Open', 'High', 'Low', 'Close', 'Volume'
apple.columns = apple.columns.get_level_values(0)

# Look at the first few rows
apple.head()

### Understanding OHLCV Data

Each row is one trading day. The columns are:

| Column | Meaning |
|--------|---------|
| **Open** | Price when the market opened (9:30 AM ET) |
| **High** | Highest price during the day |
| **Low** | Lowest price during the day |
| **Close** | Price when the market closed (4:00 PM ET) -- this is "the" stock price |
| **Volume** | Number of shares traded that day |

Together these are called **OHLCV** data -- the fundamental building block of financial analysis.

In [ ]:
# How much data do we have?
print(f"Date range: {apple.index[0].date()} to {apple.index[-1].date()}")
print(f"Number of trading days: {len(apple)}")
print(f"\nLatest closing price: ${apple['Close'].iloc[-1]:.2f}")

---

## Part 2: Visualizing Price Data

### Simple Price Chart

Let's start with the most basic chart -- the closing price over time.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(apple.index, apple['Close'], color='steelblue', linewidth=1.5)
plt.title('Apple (AAPL) Closing Price -- Past Year', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.tight_layout()
plt.show()

### Candlestick Charts

A **candlestick chart** shows all four prices (Open, High, Low, Close) for each day:

- **Green candle**: Close > Open (price went **up** that day)
- **Red candle**: Close < Open (price went **down** that day)
- The **body** shows the range between Open and Close
- The **wicks** (thin lines) show the High and Low

Traders use candlestick charts because they pack much more information than a simple line chart.

We'll use **Plotly** to make an interactive candlestick chart.

In [ ]:
# Interactive candlestick chart -- last 3 months of data
recent = apple.last('3M')

fig = go.Figure(data=[go.Candlestick(
    x=recent.index,
    open=recent['Open'],
    high=recent['High'],
    low=recent['Low'],
    close=recent['Close'],
    name='AAPL'
)])

fig.update_layout(
    title='Apple (AAPL) Candlestick Chart -- Last 3 Months',
    yaxis_title='Price ($)',
    xaxis_title='Date',
    xaxis_rangeslider_visible=True,
    template='plotly_white',
    height=500
)

fig.show()

**Try it:** Hover over candles to see the exact OHLC values. Use the range slider at the bottom to zoom into specific date ranges.

---

### Volume Chart

**Volume** tells you how many shares were traded. High volume often accompanies big price moves (earnings announcements, news events, etc.).

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), 
                                gridspec_kw={'height_ratios': [3, 1]}, 
                                sharex=True)

# Price chart on top
ax1.plot(apple.index, apple['Close'], color='steelblue', linewidth=1.5)
ax1.set_ylabel('Price ($)')
ax1.set_title('Apple (AAPL) Price and Volume', fontsize=14)

# Volume chart on bottom
ax2.bar(apple.index, apple['Volume'], color='gray', alpha=0.6, width=1)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date')

plt.tight_layout()
plt.show()

---

## Part 3: Returns -- Measuring Gains and Losses

The **return** on a stock tells you how much it gained or lost as a percentage:

$$\text{Return}_t = \frac{\text{Price}_t - \text{Price}_{t-1}}{\text{Price}_{t-1}}$$

In pandas, `pct_change()` does this calculation automatically.

In [ ]:
# Calculate daily returns
apple['Return'] = apple['Close'].pct_change()

# Show the last 10 days of prices and returns
apple[['Close', 'Return']].tail(10)

In [ ]:
# Plot daily returns
plt.figure(figsize=(12, 5))
plt.bar(apple.index, apple['Return'], color=apple['Return'].apply(
    lambda x: 'green' if x >= 0 else 'red'), alpha=0.7, width=1)
plt.axhline(y=0, color='black', linewidth=0.5)
plt.title('Apple (AAPL) Daily Returns', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Daily Return')
plt.tight_layout()
plt.show()

**What you're seeing:**
- Returns fluctuate around zero -- like a seismograph
- Most days are small moves
- Occasionally there are big spikes (earnings, news, market events)
- This "bounciness" is **volatility** -- a key measure of risk

### Summary Statistics

Let's compute the key numbers that describe Apple's performance.

In [ ]:
# Daily statistics
avg_daily_return = apple['Return'].mean()
daily_std = apple['Return'].std()
best_day = apple['Return'].max()
worst_day = apple['Return'].min()
total_return = (apple['Close'].iloc[-1] / apple['Close'].iloc[0]) - 1

# Annualized statistics (252 trading days per year)
annual_return = avg_daily_return * 252
annual_volatility = daily_std * np.sqrt(252)

print("Apple (AAPL) -- Summary Statistics")
print("=" * 50)
print(f"\nDaily:")
print(f"  Average daily return:  {avg_daily_return:.4f} ({avg_daily_return*100:.2f}%)")
print(f"  Daily std deviation:   {daily_std:.4f} ({daily_std*100:.2f}%)")
print(f"  Best day:              {best_day:.4f} ({best_day*100:.2f}%)")
print(f"  Worst day:             {worst_day:.4f} ({worst_day*100:.2f}%)")
print(f"\nAnnualized:")
print(f"  Annualized return:     {annual_return:.4f} ({annual_return*100:.2f}%)")
print(f"  Annualized volatility: {annual_volatility:.4f} ({annual_volatility*100:.2f}%)")
print(f"\nTotal return over period: {total_return*100:.2f}%")
print(f"  $1,000 invested became: ${1000*(1+total_return):,.2f}")

### Return Distribution

A histogram shows how returns are distributed. Most days cluster near zero, with "fat tails" of extreme moves.

In [ ]:
plt.figure(figsize=(10, 5))
apple['Return'].dropna().hist(bins=50, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(x=0, color='black', linewidth=1)
plt.axvline(x=avg_daily_return, color='red', linewidth=2, linestyle='--', label=f'Mean = {avg_daily_return*100:.2f}%')
plt.title('Distribution of Apple Daily Returns', fontsize=14)
plt.xlabel('Daily Return')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

---

## Part 4: Moving Averages

A **moving average** smooths out daily noise to reveal the underlying trend.

- **20-day MA**: Average of the last 20 closing prices (short-term trend)
- **50-day MA**: Medium-term trend

Each day, the window "moves" forward -- drop the oldest price, add the newest.

**Trading signals:**
- Price **above** the MA = uptrend (bullish)
- Price **below** the MA = downtrend (bearish)
- Price **crosses** the MA = potential trend reversal

In [ ]:
# Calculate moving averages
apple['MA20'] = apple['Close'].rolling(window=20).mean()
apple['MA50'] = apple['Close'].rolling(window=50).mean()

# Show a few rows -- note the first 19/49 rows are NaN (not enough data yet)
apple[['Close', 'MA20', 'MA50']].tail(10)

In [ ]:
# Calculate 20-day moving average
apple['MA20'] = apple['Close'].rolling(window=20).mean()

# Plot price versus moving average
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(apple['Close'], label='Price', linewidth=2)
plt.plot(apple['MA20'], label='20-day MA', linewidth=2, linestyle='--')
plt.legend()
plt.title('Apple Stock Price vs 20-Day Moving Average')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.show()


In [ ]:
# Plot price with moving averages
plt.figure(figsize=(12, 6))
plt.plot(apple.index, apple['Close'], label='Price', color='steelblue', linewidth=1.5)
plt.plot(apple.index, apple['MA20'], label='20-day MA', color='orange', linewidth=1.5, linestyle='--')
plt.plot(apple.index, apple['MA50'], label='50-day MA', color='red', linewidth=1.5, linestyle='--')
plt.title('Apple (AAPL) Price with Moving Averages', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

**Observations:**
- The moving averages are **smoother** than the raw price
- The 50-day MA is **smoother** than the 20-day MA (longer window = more smoothing)
- The MAs **lag** behind the price (they're based on past data)
- When the price crosses above/below a MA, traders see that as a signal

This is a simple example of a **trading signal**. In later weeks, we'll see how AI can find much more complex patterns.

### Interactive Price Chart with Moving Averages

Let's build an interactive version with Plotly so you can zoom and hover.

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=apple.index, y=apple['Close'], name='Price',
                         line=dict(color='steelblue', width=1.5)))
fig.add_trace(go.Scatter(x=apple.index, y=apple['MA20'], name='20-day MA',
                         line=dict(color='orange', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=apple.index, y=apple['MA50'], name='50-day MA',
                         line=dict(color='red', width=1.5, dash='dash')))

fig.update_layout(
    title='Apple (AAPL) Price with Moving Averages',
    yaxis_title='Price ($)',
    xaxis_title='Date',
    template='plotly_white',
    height=500,
    hovermode='x unified'
)

fig.show()

---

## Part 5: The Sharpe Ratio

How do we compare investments with different risk levels? The **Sharpe Ratio** answers: *"How much extra return do I get per unit of risk?"*

$$\text{Sharpe Ratio} = \frac{\text{Return} - \text{Risk-Free Rate}}{\text{Volatility}}$$

| Sharpe | Interpretation |
|--------|----------------|
| < 0.5 | Poor |
| 0.5 -- 1.0 | Acceptable |
| 1.0 -- 2.0 | Good |
| > 2.0 | Excellent (or suspicious!) |

In [ ]:
# Calculate the Sharpe Ratio for Apple
# We'll assume a risk-free rate of ~5% (approximate current T-bill rate)
risk_free_rate = 0.05

sharpe = (annual_return - risk_free_rate) / annual_volatility

print(f"Annualized return:     {annual_return*100:.2f}%")
print(f"Risk-free rate:        {risk_free_rate*100:.2f}%")
print(f"Annualized volatility: {annual_volatility*100:.2f}%")
print(f"\nSharpe Ratio: {sharpe:.2f}")

if sharpe < 0.5:
    print("Interpretation: Poor risk-adjusted performance")
elif sharpe < 1.0:
    print("Interpretation: Acceptable risk-adjusted performance")
elif sharpe < 2.0:
    print("Interpretation: Good risk-adjusted performance")
else:
    print("Interpretation: Excellent risk-adjusted performance")

---

## Part 6: Comparing Two Stocks

Let's compare Apple with Microsoft to see how two tech stocks stack up.

In [ ]:
# Download both stocks
tickers = ['AAPL', 'MSFT']
data = yf.download(tickers, period='1y')

# Extract closing prices (multi-ticker download keeps MultiIndex)
prices = data['Close']
prices.head()

In [ ]:
# Normalize prices to start at $100 for easy comparison
# This shows cumulative performance on the same scale
normalized = (prices / prices.iloc[0]) * 100

plt.figure(figsize=(12, 6))
for col in normalized.columns:
    plt.plot(normalized.index, normalized[col], linewidth=1.5, label=col)

plt.axhline(y=100, color='gray', linewidth=0.5, linestyle='--')
plt.title('Normalized Price Comparison: AAPL vs MSFT (Starting at $100)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Normalized Price ($)')
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate returns for both
returns = prices.pct_change().dropna()

# Summary comparison table
summary = pd.DataFrame({
    'Avg Daily Return': returns.mean(),
    'Daily Std Dev': returns.std(),
    'Ann. Return': returns.mean() * 252,
    'Ann. Volatility': returns.std() * np.sqrt(252),
    'Sharpe Ratio': (returns.mean() * 252 - risk_free_rate) / (returns.std() * np.sqrt(252)),
    'Total Return': (prices.iloc[-1] / prices.iloc[0]) - 1
})

# Format nicely
summary_display = summary.copy()
for col in summary_display.columns:
    if col == 'Sharpe Ratio':
        summary_display[col] = summary_display[col].map('{:.2f}'.format)
    else:
        summary_display[col] = summary_display[col].map('{:.2%}'.format)

print("Stock Comparison: AAPL vs MSFT")
print("=" * 50)
summary_display.T

In [ ]:
# Correlation between the two stocks
correlation = returns['AAPL'].corr(returns['MSFT'])
print(f"Correlation between AAPL and MSFT daily returns: {correlation:.3f}")
print(f"\nInterpretation: A correlation of {correlation:.2f} means the stocks tend to")
if correlation > 0.7:
    print("move strongly together (highly correlated).")
elif correlation > 0.4:
    print("move somewhat together (moderately correlated).")
else:
    print("move somewhat independently (low correlation).")

In [ ]:
# Scatter plot of returns
plt.figure(figsize=(7, 7))
plt.scatter(returns['AAPL'], returns['MSFT'], alpha=0.4, s=15, color='steelblue')
plt.axhline(y=0, color='gray', linewidth=0.5)
plt.axvline(x=0, color='gray', linewidth=0.5)
plt.xlabel('AAPL Daily Return')
plt.ylabel('MSFT Daily Return')
plt.title(f'Return Scatter Plot (Correlation = {correlation:.3f})', fontsize=14)
plt.tight_layout()
plt.show()

---

## Part 7: Your Turn -- Exercises

Now it's your turn to apply what you've learned.

### Exercise 1: Analyze a Stock of Your Choice

Pick a stock you're interested in. Download its data, calculate returns, compute moving averages, and calculate the Sharpe Ratio.

Some ticker ideas: `TSLA`, `NVDA`, `AMZN`, `GOOG`, `JPM`, `GS`, `XOM`, `META`

In [ ]:
# YOUR CODE HERE
# 1. Download data for a ticker of your choice
# 2. Flatten the MultiIndex columns
# 3. Calculate daily returns
# 4. Calculate 20-day and 50-day moving averages
# 5. Plot the price with moving averages
# 6. Calculate and print the Sharpe Ratio



### Exercise 2: When Do Moving Averages Cross?

A popular trading signal is the **"Golden Cross"** -- when the short-term MA crosses *above* the long-term MA (bullish signal). The opposite is a **"Death Cross"** (bearish).

Using the Apple data from earlier, find the dates where the 20-day MA crossed the 50-day MA.

In [ ]:
# YOUR CODE HERE
# Hint: A crossover happens when (MA20 - MA50) changes sign.
# You can detect sign changes with:
#   apple['Signal'] = apple['MA20'] - apple['MA50']
#   crossovers = apple[apple['Signal'] * apple['Signal'].shift(1) < 0]



---

## Part 8: Prompt Engineering for Finance

Now that you've worked with financial data hands-on, let's practice using AI to help with analysis.

**Prompt engineering** = writing better instructions to get better AI outputs.

### Principles:
1. **Be specific**: Instead of "analyze this," say "calculate the Sharpe ratio"
2. **Provide context**: "I'm a beginner" vs. "I'm preparing a board presentation"
3. **Give examples**: Show the AI what format you want
4. **Include data**: Paste actual numbers from your analysis
5. **Iterate**: If the first response isn't right, refine your prompt

### Exercise 3: Explain a Concept

**Task:** Write a prompt asking an AI (ChatGPT or Claude) to explain what a moving average crossover means and why traders care about it.

**Bad prompt:** "What is a moving average?"

**Write a better prompt below, test it, and paste the response:**

```
YOUR PROMPT HERE:



```

**AI Response:** (Paste response here)

```



```

**Your evaluation:** Was the response helpful? What made your prompt effective (or not)?

### Exercise 4: Interpret Your Results

**Task:** Take the summary statistics you calculated above for Apple (or your chosen stock) and write a prompt asking the AI to interpret them.

**Tips:**
- Include the actual numbers in your prompt
- Ask for a comparison to the S&P 500 (historically ~10%/year return, ~15% volatility)
- Specify your audience (e.g., "explain to someone with no finance background")

```
YOUR PROMPT HERE:



```

**AI Response:** (Paste response here)

```



```

---

## Part 9: Reflection

Answer these questions based on your experience today:

### 1. What did you learn about working with financial data in Python?

*Your answer:*


### 2. Which chart type (line, candlestick, histogram) did you find most informative and why?

*Your answer:*


### 3. What are the limitations of using AI for financial analysis?

*Your answer:*
